# Serve Jarida from Kaggle

Runs the whole RAG on Kaggle's GPUs and opens a public HTTPS address for the frontend, free.
The address changes every run: paste it in the app under **Settings → Connection**.

**Before running**
- Settings: **Accelerator GPU T4 x2**, **Internet on**.
- Add-ons → Secrets: `NEON_DATABASE_URL` and `HF_TOKEN` (the same ones `generation.ipynb` uses).
- The Drive folder `jarida` is shared as "Anyone with the link → Viewer", so new PDFs can be indexed.

**To keep it up for hours**, use *Save Version → Save & Run All*: the last cell holds the server open
until the session ends (Kaggle stops it after 12 hours). The address is printed in the version's log.

In [ ]:
!git clone -q https://github.com/cad0m/jarida-rag.git || git -C jarida-rag pull -q
!pip install -q -e "jarida-rag[serve]"

In [ ]:
import os
import re
import subprocess
import sys
import threading
import time

from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
os.environ["DATABASE_URL"] = secrets.get_secret("NEON_DATABASE_URL")
os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")
os.environ["JARIDA_WORK"] = "/kaggle/working"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"  # must be set before torch loads
# the editable install is only on the path after a restart, so add it for this session
sys.path.insert(0, "/kaggle/working/jarida-rag/src")

import mlflow
import requests
import uvicorn
from jarida_rag import api, config, rag, store

mlflow.set_tracking_uri(config.MLFLOW_URI)  # every answer is traced locally, in /kaggle/working/mlflow.db

## Load the models and bring the index up to date

`update_index` embeds only the stored PDFs that are new or were indexed with other settings, so after the
first run it usually has nothing to do.

In [ ]:
pool = store.connect()
embedder = store.load_embedder()
reranker = store.load_reranker()
print(store.update_index(pool, embedder), "documents indexed now")
lexical_index = store.load_lexical(pool) if config.HYBRID else None
tokenizer, model = rag.load_llm()
print(f"LLM on {config.LLM_DEVICE}: {model.get_memory_footprint() / 1024**3:.1f} GB")

In [ ]:
# smoke test before going public: one question straight through the pipeline
reply, hits = rag.answer("ما هو المرسوم رقم 2.24.830؟", pool, embedder, tokenizer, model, reranker, lexical_index)
print(reply[:500])
print(len(hits), "sources:", [api.sources(pool, hits)[0][field] for field in ("issue", "title", "drive_id")])

## Start the API and the public tunnel

The API listens on port 8000 inside the notebook. `cloudflared` opens a free Cloudflare "quick tunnel" to it,
no account needed, and prints an `https://….trycloudflare.com` address.

In [ ]:
server = uvicorn.Server(uvicorn.Config(api.create_app(pool, embedder, reranker, tokenizer, model, lexical_index), host="0.0.0.0", port=8000, log_level="warning"))
threading.Thread(target=server.run, daemon=True).start()
time.sleep(3)
print(requests.get("http://localhost:8000/health", timeout=10).json())

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared

tunnel = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://localhost:8000", "--no-autoupdate"],
    stderr=subprocess.PIPE, text=True,
)
API_URL = None
for line in tunnel.stderr:
    found = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if found:
        API_URL = found.group(0)
        break

time.sleep(5)  # the new address takes a few seconds to resolve
print("API URL:", API_URL)
print("health through the tunnel:", requests.get(f"{API_URL}/health", timeout=30).json())
print()
print("Paste the API URL in the app: Settings → Connection → Test.")

## Keep it running

Interactive sessions stop when idle; a saved version (*Save & Run All*) keeps going until this cell is stopped
or the 12-hour limit is reached. The log line every 10 minutes shows the server is alive.

In [ ]:
while True:
    time.sleep(600)
    try:
        status = requests.get("http://localhost:8000/health", timeout=10).json()
    except requests.RequestException as error:
        status = f"health check failed: {error}"
    print(time.strftime("%H:%M"), API_URL, status, flush=True)